# Agrupamento

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/03_clustering.ipynb)

- **Objetivo:** agrupar os enunciados com k-means, projetar as representações com UMAP e escolher k pela silhueta.
- **Entradas:** representações salvas pelo `02_embeddings`; áreas e temas do JurisTCU.
- **Saídas:** tabelas de atribuições aos grupos, da silhueta e das projeções UMAP em `results/tables/`.
- **Requisitos:** não requer GPU. A varredura de k leva horas em CPU e fica no fim do notebook.
- Áreas e temas definem apenas k e o filtro de tema. A avaliação com rótulos fica no notebook 04.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import importlib

import numpy as np
import pandas as pd

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import clustering, data, evaluation, storage  # noqa: E402
from juristcu.config import (  # noqa: E402
    AREA_COLUMN,
    HDBSCAN_MIN_CLUSTER_SIZE,
    K_RANGE,
    MIN_TOPIC_SIZE,
    N_DOCUMENTS,
    REPRESENTATION_NAMES,
    SEED,
    TOPIC_COLUMN,
)

storage.save_environment("03")

## 1. Carregamento

- Áreas e temas do JurisTCU. Servem apenas para definir k e o filtro de tema.
- Representações salvas pelo `02_embeddings`, na ordem dos documentos. R3 aparece uma vez por semente.
- Conferência: uma linha por documento e norma 1 em cada linha.

In [ ]:
documents = data.load_documents(columns=(AREA_COLUMN, TOPIC_COLUMN))
topics = documents[TOPIC_COLUMN].to_numpy()
n_areas = documents[AREA_COLUMN].nunique()
# O protocolo conta 10 áreas.
assert n_areas == 10
representations = {name: storage.load_embeddings(name) for name in REPRESENTATION_NAMES}

In [ ]:
check = pd.DataFrame(
    [
        {
            "representação": name,
            "linhas": x.shape[0],
            "dimensão": x.shape[1],
            "desvio máximo da norma": float(np.abs(np.linalg.norm(x, axis=1) - 1).max()),
        }
        for name, x in representations.items()
    ]
)
assert (check["linhas"] == N_DOCUMENTS).all()
assert (check["desvio máximo da norma"] < 1e-4).all()
check

## 2. k-means por área

- k igual ao número de áreas (10), com a mesma semente para todas as representações.
- A tabela `area_assignments` guarda o grupo de cada documento, uma coluna por representação.
- A coluna `document` é a posição do documento no arquivo.

In [ ]:
area_preds = {
    name: clustering.run_kmeans(x, k=n_areas, seed=SEED) for name, x in representations.items()
}
area_assignments = clustering.assignments_table(area_preds, np.arange(len(documents)))
storage.save_table(area_assignments, "area_assignments")
area_assignments.head()

## 3. k-means por tema

- Mantém apenas os temas com ao menos 20 documentos, antes do k-means.
- k igual ao número de temas mantidos (144), com a mesma semente.
- A tabela `topic_assignments` contém só os documentos mantidos.
- A coluna `document` aponta a posição de cada documento mantido no arquivo.

In [ ]:
topic_mask = evaluation.filter_min_size(topics, min_size=MIN_TOPIC_SIZE)
n_topics = documents.loc[topic_mask, TOPIC_COLUMN].nunique()
# O protocolo conta 144 temas com ao menos 20 documentos.
assert n_topics == 144
int(topic_mask.sum()), n_topics

In [ ]:
topic_preds = {
    name: clustering.run_kmeans(x[topic_mask], k=n_topics, seed=SEED)
    for name, x in representations.items()
}
topic_assignments = clustering.assignments_table(topic_preds, np.flatnonzero(topic_mask))
storage.save_table(topic_assignments, "topic_assignments")
topic_assignments.head()

## 4. Silhueta dos agrupamentos principais

- Silhueta do k-means por área e por tema, em cada representação.
- Usa a representação original e todos os documentos da análise, sem subamostra.
- A tabela `cluster_silhouette` tem uma linha por representação e análise.

In [ ]:
# Documentos, agrupamentos e k de cada análise.
analyses = {
    "area": (np.ones(len(documents), dtype=bool), area_preds, n_areas),
    "topic": (topic_mask, topic_preds, n_topics),
}
cluster_silhouette = pd.DataFrame(
    [
        {
            "representation": name,
            "analysis": analysis,
            "k": k,
            "silhouette": evaluation.internal_score(x[mask], preds[name], seed=SEED),
        }
        for analysis, (mask, preds, k) in analyses.items()
        for name, x in representations.items()
    ]
)
storage.save_table(cluster_silhouette, "cluster_silhouette")
cluster_silhouette

## 5. Projeções UMAP

- Projeção de cada representação em 2 dimensões, com a mesma semente.
- Uma tabela por representação (`umap_<nome>`), com as colunas x e y, na ordem dos documentos.
- A projeção serve à visualização e ao HDBSCAN. A silhueta usa a representação original.

In [ ]:
projections = {}
for name, x in representations.items():
    projections[name] = clustering.project_umap(x, seed=SEED)
    storage.save_table(pd.DataFrame(projections[name], columns=["x", "y"]), f"umap_{name}")

## 6. HDBSCAN (opcional)

- HDBSCAN sobre a projeção UMAP. O número de grupos não é fixado.
- Primeiro item da ordem de corte, em caso de atraso no cronograma.
- O resumo conta os grupos sem o ruído (-1) e mede a fração de ruído.
- Não usa rótulos. NMI e ARI ficam para o notebook 04, apenas nos documentos fora do ruído.

In [ ]:
hdbscan_preds = {
    name: clustering.run_hdbscan(projection, min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE)
    for name, projection in projections.items()
}
hdbscan_assignments = clustering.assignments_table(hdbscan_preds, np.arange(len(documents)))
storage.save_table(hdbscan_assignments, "hdbscan_assignments")

In [ ]:
# O valor -1 indica ruído e não conta como grupo.
hdbscan_summary = pd.DataFrame(
    [
        {
            "representation": name,
            "n_groups": len(np.setdiff1d(preds, [-1])),
            "noise_share": float(np.mean(preds == -1)),
        }
        for name, preds in hdbscan_preds.items()
    ]
)
storage.save_table(hdbscan_summary, "hdbscan_summary")
hdbscan_summary

## 7. Número de grupos pela silhueta

- k de 5 a 50, sem consultar áreas nem temas. Silhueta com todos os documentos.
- A varredura leva horas em CPU. Por isso fica no fim do notebook.
- A tabela `silhouette_by_k` é regravada após cada representação.
- Ao reexecutar, as representações já concluídas são carregadas e puladas.
- Para refazer do zero, apagar `results/tables/silhouette_by_k.csv`.
- No Colab, o disco é apagado ao fim da sessão. Para retomar em outra sessão, baixar o arquivo parcial e enviá-lo de volta para `results/tables/` pelo painel **Arquivos**.
- A tabela `best_k` compara o k de maior silhueta com o número de áreas.

In [ ]:
try:
    saved = storage.load_table("silhouette_by_k")
except FileNotFoundError:
    saved = clustering.sweep_table({})
# Reaproveita apenas as representações com todos os valores de k.
sweeps = clustering.completed_sweeps(saved, REPRESENTATION_NAMES, K_RANGE)
list(sweeps)

In [ ]:
for name in REPRESENTATION_NAMES:
    if name not in sweeps:
        sweeps[name] = clustering.silhouette_sweep(representations[name], K_RANGE, seed=SEED)
        # Regrava a tabela na ordem de REPRESENTATION_NAMES.
        sweeps = {key: sweeps[key] for key in REPRESENTATION_NAMES if key in sweeps}
        storage.save_table(clustering.sweep_table(sweeps), "silhouette_by_k")

In [ ]:
best_k = clustering.best_k_table(sweeps, n_areas)
storage.save_table(best_k, "best_k")
best_k

## 8. Conferência final

- Relê as tabelas gravadas e confere o número de linhas.
- As tabelas de atribuições têm a coluna `document` e uma coluna por representação.
- O HDBSCAN fica fora da conferência, por ser opcional.

In [ ]:
n_representations = len(REPRESENTATION_NAMES)
# Linhas esperadas em cada tabela gravada.
expected_rows = {
    "area_assignments": N_DOCUMENTS,
    "topic_assignments": int(topic_mask.sum()),
    "cluster_silhouette": 2 * n_representations,
    **{f"umap_{name}": N_DOCUMENTS for name in REPRESENTATION_NAMES},
    "silhouette_by_k": n_representations * len(K_RANGE),
    "best_k": n_representations,
}
tables = {name: storage.load_table(name) for name in expected_rows}
summary = pd.DataFrame(
    [
        {
            "tabela": name,
            "linhas": len(table),
            "linhas esperadas": expected_rows[name],
            "colunas": table.shape[1],
        }
        for name, table in tables.items()
    ]
)
assert (summary["linhas"] == summary["linhas esperadas"]).all()
assert tables["area_assignments"].shape[1] == 1 + n_representations
assert tables["topic_assignments"].shape[1] == 1 + n_representations
summary